# 2D Optimization Training with TensorBoard

This notebook demonstrates how to run the optimization loop using the `OptimizationTrainer` and visualize progress using TensorBoard directly within the notebook.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import joblib
import numpy as np
import torch

from diffmeshopt.opt2d.optimize import ContourRefiner
from diffmeshopt.opt2d.props import OptimizationProps, SamplingProps, TemplateProps
from diffmeshopt.opt2d.trainer import OptimizationTrainer

In [5]:
# 1. Load Data
data_path = Path("/workspace/diffmeshopt/data/2d_training_data.pkl")
if not data_path.exists():
    print(f"Data file {data_path} not found. Please run src/generate_2d_data.py first.")
else:
    print(f"Loading data from {data_path}...")
    data = joblib.load(data_path)
    image_np = -data["image"]  # Invert intensity if needed based on previous notebooks
    contour_np = data["contour"]
    gt_contour_np = data["gt"]

Loading data from /workspace/diffmeshopt/data/2d_training_data.pkl...


In [ ]:
# 2. Setup Refiner
opt_props = OptimizationProps(lr=0.05)
sampl_props = SamplingProps(num_samples=51, sample_length=1.0)
template_props = TemplateProps(sigma=2.0, peak_dist=5.0, num_samples=51)

refiner = ContourRefiner(
    image=image_np,
    initial_contour=contour_np,
    optimization_props=opt_props,
    sampling_props=sampl_props,
    template_props=template_props,
    template_mode="global",
)

In [13]:
# 3. Setup Trainer
output_dir = Path("/workspace/diffmeshopt/output/experiment_tb")

trainer = OptimizationTrainer(
    refiner=refiner,
    output_dir=output_dir,
    gt_contour=gt_contour_np,
    image=image_np,
    num_iterations=500,
    save_interval=100,
    log_interval=10,
    log_image_interval=50,
    use_tensorboard=True,
)

In [14]:
# 4. Launch TensorBoard
# Load the TensorBoard notebook extension
%load_ext tensorboard
# Start TensorBoard pointing to the logs directory
%tensorboard --logdir /workspace/diffmeshopt/output/experiment_tb/logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 61300), started 0:02:29 ago. (Use '!kill 61300' to kill it.)

In [ ]:
# 5. Run Training
trainer.fit()

Optimizing:   0%|          | 0/500 [00:00<?, ?it/s]

torch.Size([100, 200]) torch.Size([21])


RuntimeError: No active exception to reraise